In [1]:
%matplotlib inline
import casadi as ca
import l4casadi as l4c
import numpy as np
import torch
from acados_template import AcadosModel
import matplotlib.pyplot as plt

from simple_neural_mpc.neural_modeling.dataset.datamodule import Datamodule
from simple_neural_mpc.neural_modeling.dataset.tensor_dataset import TensorDataset
from simple_neural_mpc.utils import project_root
import numpy as np
from simple_neural_mpc.config.neural_config import DatasetConfig
from simple_neural_mpc.config.neural_config import TrainerConfig
from simple_neural_mpc.neural_modeling.learner.next_state_learner import NextStateLearner 
from simple_neural_mpc.neural_modeling.learner.mlp import MLP     
from simple_neural_mpc.neural_modeling.learner.trainer import Trainer
from torch.nn import ReLU, MSELoss

/home/brock/Desktop/simple_neural_mpc/simple_neural_mpc/neural_modeling/learner/mlp.py:8: SyntaxWarning: invalid escape sequence '\d'
  """


In [2]:
# UAV MODEL
X = np.load("../../../X.npy") # [x, y, z, vx, vy, vz, r, p, y, r_c, p_c, y_c, thrust, t] = [state, controls, time]
Y = np.load("../../../Y.npy") # [x, y, z, vx, vy, vz, r, p, y, num_traj]

print(X.shape, Y.shape)

(50224, 14) (50224, 10)


In [ ]:
ax = plt.figure(figsize=(8, 8)).add_subplot(projection='3d')
ax.scatter(Y[:, 0], Y[:, 1], Y[:, 2], s=0.4, color="forestgreen")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

In [3]:
dataset = TensorDataset(torch.from_numpy(X).to(torch.float32), torch.from_numpy(Y).to(torch.float32))
TrainerConfig.wandb_project = f"{TrainerConfig.wandb_project}_{DatasetConfig.name}"
datamodule = Datamodule(dataset, savedpath=None)

In [ ]:
learner = NextStateLearner(
    is_pinn = False,
    state_dim = 10,
    input_dim = 4,
    in_mpc = False
)
trainer = Trainer()
trainer.fit(learner, datamodule)

wandb: Currently logged in as: neverorfrog (neverorfrog-sapienza). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/brock/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/brock/miniconda3/envs/uav_env/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/home/brock/miniconda3/envs/uav_env/lib/python3.13/site-packages/lightning/pytorch/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.

  | Name | Type    | Params | Mode 
-----------------------------------------
0 | mlp  | MLP     | 137 K  | train
1 | mse  | MSELoss | 0      | train
-----------------------------------------
137 K     Trainable params
0         Non-trainable params
137 K     Total params
0.552     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


/home/brock/miniconda3/envs/uav_env/lib/python3.13/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32, 10])) that is different to the input size (torch.Size([1, 32, 10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0: 100%|██████████| 1099/1099 [00:06<00:00, 179.24it/s, v_num=4dr6, train/imit_loss_step=12.00, train/loss_step=12.00]   

/home/brock/miniconda3/envs/uav_env/lib/python3.13/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([20, 10])) that is different to the input size (torch.Size([1, 20, 10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


...

Epoch 32: 100%|██████████| 1099/1099 [00:10<00:00, 102.57it/s, v_num=4dr6, train/imit_loss_step=0.222, train/loss_step=0.222, val/imit_loss_step=4.060, val/loss_step=4.060, val/imit_loss_epoch=1.900, val/loss_epoch=1.900, train/imit_loss_epoch=13.20, train/loss_epoch=13.20]

Metric val/loss improved by 2.018 >= min_delta = 1e-06. New best score: 1.904


Epoch 47: 100%|██████████| 1099/1099 [00:10<00:00, 103.51it/s, v_num=4dr6, train/imit_loss_step=2.140, train/loss_step=2.140, val/imit_loss_step=5.950, val/loss_step=5.950, val/imit_loss_epoch=4.120, val/loss_epoch=4.120, train/imit_loss_epoch=12.90, train/loss_epoch=12.90] 

Monitored metric val/loss did not improve in the last 15 records. Best score: 1.904. Signaling Trainer to stop.


Epoch 47: 100%|██████████| 1099/1099 [00:10<00:00, 103.43it/s, v_num=4dr6, train/imit_loss_step=2.140, train/loss_step=2.140, val/imit_loss_step=5.950, val/loss_step=5.950, val/imit_loss_epoch=4.120, val/loss_epoch=4.120, train/imit_loss_epoch=12.90, train/loss_epoch=12.90]
